In [0]:
# importando funcoes 

from pyspark.sql.functions import col, split, regexp_replace, trim, expr, when, lower, translate



In [0]:
# Lendo a tabela bronze

df_silver_stage01 = spark.read.table("tabela_despesas_bronze")

In [0]:

# Removendo o R$ e os espaços, trocando vírgula por ponto, fatiando o primeiro espaço, transformando em 2 individuos.

df_silver_stage01 = df_silver_stage01.withColumn("texto_limpo", trim(regexp_replace(col("descricao_valor"), r"R\$\s*", ""))) \
    .withColumn("texto_ponto", regexp_replace(col("texto_limpo"), ",", ".")).withColumn("fatias", split(col("texto_ponto"), " ", 2)) \
        .withColumn("fatias", split(col("texto_ponto"), " ", 2))
    
    


In [0]:
# Pegando a primeira fatia de forma segura em seguida converteos para numero, se for texto/data vira NULL, Filtra e joga fora os NULLs (linhas de totais/lixo).

df_silver_limpo = df_silver_stage01.withColumn("primeira_fatia", expr("get(fatias, 0)")) \
    .withColumn("valor_gasto", expr("try_cast(primeira_fatia AS double)")) \
    .filter(col("valor_gasto").isNotNull())

In [0]:

# Selecionamos apenas as colunas as quais vamos usar daqui em diante.

df_silver_limpo = df_silver_limpo.select(
        col("data"),
        col("valor_gasto"),
        expr("get(fatias, 1)").alias("categoria")
    )




In [0]:
# Convertendo todas as linhas do dataframe para minúsculo possibilitando simplificar a pesquisa.

df_silver_final = df_silver_limpo.withColumn("cat_minuscula", lower(col("categoria")))



In [0]:
# Aplicando as regras de negócio baseadas no seu distinct
df_silver_final = df_silver_final.withColumn(
    "categoria_padronizada",
    when(col("cat_minuscula").rlike("entregas|babidi"), "Entregas")
    .when(col("cat_minuscula").rlike("gas|gás"), "Gás")
    .when(col("cat_minuscula").rlike("máquina|cartão"), "Taxas Máquina de Cartão")
    .when(col("cat_minuscula").rlike("bruno"), "Pagamento - Bruno")
    .when(col("cat_minuscula").rlike("wanderson"), "Pagamento - Wanderson")
    .when(col("cat_minuscula").rlike("marketing|instagram|will"), "Marketing")
    .when(col("cat_minuscula").rlike("queijo"), "Insumos - Queijo")
    .when(col("cat_minuscula").rlike("supermercado"), "Insumos - Supermercado")
    .when(col("cat_minuscula").rlike("energia|aluguel|contador|crédito|patrocínio"), "Custos Fixos / Operacionais")
    .when(col("categoria").isNull(), "Sem Descrição")
    .otherwise("Outros") 
)

# Removendo a coluna auxiliar e verificar o resultado
df_silver_final = df_silver_final.drop("cat_minuscula")

display(df_silver_final.select("valor_gasto", "categoria", "categoria_padronizada"))

In [0]:
df_silver_final.write.mode("overwrite").format("delta").saveAsTable("tabela_silver_despesas_pizzaria")